In [11]:
# Cell 1
# Mục đích: kết nối BigQuery và định nghĩa lại các hàm dùng chung (notebook mới phải định nghĩa lại).
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

load_dotenv()                                    # đọc file .env rồi nạp vào biến môi trường
OLIST_DIR = Path(os.getenv("OLIST_DIR", ""))     # thư mục chứa 9 file CSV (ngoài repo)
PROJECT = os.getenv("GCP_PROJECT", "")
LOCATION = os.getenv("BQ_LOCATION", "US")

client = bigquery.Client(project=PROJECT, location=LOCATION)   # kênh kết nối, location khớp dataset


def mb(n):
    # đổi byte sang MB; trả None nếu n là None để lỗi "đọc byte quá sớm" lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)


def check(sql):
    # dry run: chỉ ước lượng số MB sẽ quét, không đọc dữ liệu, không tốn tiền
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Sẽ quét: {mb(job.total_bytes_processed)} MB")


def stats(sql, max_mb=100):
    # chạy thật, tắt cache, trần byte billed = max_mb; trả DataFrame kết quả
    cfg = bigquery.QueryJobConfig(
        use_query_cache=False,
        maximum_bytes_billed=max_mb * 1024 * 1024,
    )
    job = client.query(sql, job_config=cfg)      # gửi job, chưa chờ
    df = job.result().to_dataframe()             # chờ xong rồi đưa kết quả vào df
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df


print(len(list(OLIST_DIR.glob("*.csv"))), "file CSV;", PROJECT, LOCATION)

9 file CSV; bq-learning-510104 US


In [7]:
# Cell 2
# Mục đích: định nghĩa hàm nạp một file CSV thành bảng toàn STRING trong olist_raw (chưa nạp gì cả).
def load_csv_as_string(csv_path, table_name):
    # đọc đúng dòng tiêu đề (nrows=0) để lấy tên cột; utf-8-sig bỏ ký tự BOM nếu file có
    header = pd.read_csv(csv_path, nrows=0, encoding="utf-8-sig").columns.tolist()
    schema = [bigquery.SchemaField(col, "STRING") for col in header]   # mọi cột là STRING

    cfg = bigquery.LoadJobConfig(                  # chỉ là cấu hình, chưa nạp gì
        schema=schema,
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,                       # bỏ dòng tiêu đề, không để lẫn vào dữ liệu
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,   # chạy lại không bị trùng
        allow_quoted_newlines=True,                # cho phép chữ xuống dòng nằm trong ô có dấu ngoặc kép
    )

    table_id = f"{PROJECT}.olist_raw.{table_name}"
    with open(csv_path, "rb") as f:                # mở dạng byte thô, tự đóng khi hết khối with
        job = client.load_table_from_file(f, table_id, job_config=cfg)   # việc nạp xảy ra ở dòng này
    job.result()                                   # chờ job xong (không gán lại vào job)
    return job.output_rows                         # số dòng BigQuery báo đã ghi

In [ ]:
# Cell 3
# Mục đích: nạp 8 bảng và đối chiếu số dòng giữa 4 nguồn: file CSV, job BigQuery, bảng đích, số của Ngày 2.
TABLES = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_orders_dataset.csv": "orders",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
}
EXPECTED = {   # số dòng đã đếm ở Ngày 2
    "customers": 99441, "geolocation": 1000163, "order_items": 112650,
    "order_payments": 103886, "order_reviews": 99224, "orders": 99441,
    "products": 32951, "sellers": 3095,
}

print(len(TABLES), "bảng sẽ nạp")      # in trước vòng for: rỗng thì vòng chạy 0 lần mà không báo lỗi

problems = []
for csv_name, table_name in TABLES.items():
    csv_path = OLIST_DIR / csv_name
    try:
        if not csv_path.exists():
            raise FileNotFoundError(csv_name)

        # đếm dòng từ file CSV theo từng khúc 100.000 dòng (geolocation 1 triệu dòng, tránh đầy RAM)
        n_csv = 0
        for chunk in pd.read_csv(csv_path, dtype=str, chunksize=100_000):
            n_csv += len(chunk)

        n_job = load_csv_as_string(csv_path, table_name)                        # nạp thật
        n_tbl = client.get_table(f"{PROJECT}.olist_raw.{table_name}").num_rows  # không tốn byte

        ok = (n_csv == n_job == n_tbl == EXPECTED[table_name])
        print(f"{'OK  ' if ok else 'LỆCH'} {table_name:<15} csv={n_csv:>9,} job={n_job:>9,} bảng={n_tbl:>9,} ngày2={EXPECTED[table_name]:>9,}")
        if not ok:
            problems.append(table_name)
    except Exception as e:      # một bảng lỗi thì in LỖI rồi đi tiếp; nhớ tự đọc output tìm chữ LỖI
        print(f"LỖI {table_name}: {e}")
        problems.append(table_name)

print("Bảng có vấn đề:", problems)

8 bảng sẽ nạp
OK   customers       csv=   99,441 job=   99,441 bảng=   99,441 ngày2=   99,441
OK   geolocation     csv=1,000,163 job=1,000,163 bảng=1,000,163 ngày2=1,000,163
OK   order_items     csv=  112,650 job=  112,650 bảng=  112,650 ngày2=  112,650
OK   order_payments  csv=  103,886 job=  103,886 bảng=  103,886 ngày2=  103,886
OK   order_reviews   csv=   99,224 job=   99,224 bảng=   99,224 ngày2=   99,224
OK   orders          csv=   99,441 job=   99,441 bảng=   99,441 ngày2=   99,441
OK   products        csv=   32,951 job=   32,951 bảng=   32,951 ngày2=   32,951
OK   sellers         csv=    3,095 job=    3,095 bảng=    3,095 ngày2=    3,095
Bảng có vấn đề: []


In [9]:
# Cell 4
# Mục đích: kiểm số cột, kiểu cột và xem tên cột thật của 8 bảng (không tốn byte).
EXPECT_COLS = {   # số cột đã thấy ở Ngày 1
    "customers": 5, "geolocation": 5, "order_items": 7, "order_payments": 5,
    "order_reviews": 7, "orders": 8, "products": 9, "sellers": 4,
}

for table_name, n_expected in EXPECT_COLS.items():
    t = client.get_table(f"{PROJECT}.olist_raw.{table_name}")   # thông tin về bảng, không đọc dữ liệu
    names = [f.name for f in t.schema]
    types = {f.field_type for f in t.schema}                    # tập các kiểu có trong bảng
    flag = "OK  " if (len(names) == n_expected and types == {"STRING"}) else "LỆCH"
    print(f"{flag} {table_name}: {len(names)} cột, kiểu {types}")
    print("     ", names)

OK   customers: 5 cột, kiểu {'STRING'}
      ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
OK   geolocation: 5 cột, kiểu {'STRING'}
      ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
OK   order_items: 7 cột, kiểu {'STRING'}
      ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
OK   order_payments: 5 cột, kiểu {'STRING'}
      ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
OK   order_reviews: 7 cột, kiểu {'STRING'}
      ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
OK   orders: 8 cột, kiểu {'STRING'}
      ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_es

In [12]:
# Cell 5
# Mục đích: kiểm order_reviews: chữ xuống dòng trong ô có nạp nguyên vẹn không, và ô trống thành NULL hay chuỗi rỗng.
# rf"""...""": chữ f để chèn {PROJECT}; chữ r để giữ nguyên dấu \ cho BigQuery.
sql = rf"""
SELECT
  COUNT(*) AS so_dong,
  COUNTIF(REGEXP_CONTAINS(review_comment_message, r'\n')) AS o_co_xuong_dong,
  COUNTIF(review_comment_message IS NULL) AS message_null,
  COUNTIF(review_comment_message = '') AS message_rong
FROM `{PROJECT}.olist_raw.order_reviews`
"""
check(sql)           # thấy khoảng vài MB thì mới chạy tiếp
df = stats(sql)
df

Sẽ quét: 2.81 MB
processed: 2.81 MB | billed: 10.0 MB


,so_dong,o_co_xuong_dong,message_null,message_rong
0,99224,3852,58247,0


In [13]:
# Cell 5b
# Mục đích: xem độ dài mã zip ở 3 bảng có zip (số 0 đầu có còn hay đã mất từ file gốc).
sql = f"""
SELECT 'customers' AS bang, LENGTH(customer_zip_code_prefix) AS do_dai, COUNT(*) AS so_dong
FROM `{PROJECT}.olist_raw.customers` GROUP BY 2
UNION ALL
SELECT 'sellers', LENGTH(seller_zip_code_prefix), COUNT(*)
FROM `{PROJECT}.olist_raw.sellers` GROUP BY 2
UNION ALL
SELECT 'geolocation', LENGTH(geolocation_zip_code_prefix), COUNT(*)
FROM `{PROJECT}.olist_raw.geolocation` GROUP BY 2
ORDER BY bang, do_dai
"""
check(sql)           # 3 bảng: mong đợi khoảng 30 MB billed
df = stats(sql)
df

Sẽ quét: 7.36 MB
processed: 7.36 MB | billed: 30.0 MB


,bang,do_dai,so_dong
0,customers,5,99441
1,geolocation,5,1000163
2,sellers,5,3095
